# 🚀 BuildFail-SLM: High-Speed Fine-Tuning on Kaggle (Dual T4 GPUs - 32 GB VRAM)

Fine-tunes **`Qwen/Qwen2.5-Coder-3B-Instruct`** across all **32,879 failure cases** on **Kaggle's Dual Tesla T4 GPUs (32 GB total VRAM)**.

### ⚠️ REQUIRED KAGGLE SETTINGS (Right Sidebar):
1. **Accelerator**: Select **GPU T4 x2** (Dual GPUs = 32 GB VRAM)
2. **Internet**: Toggle to **Internet On** (Required to download dataset and push adapter to Hugging Face)
3. **Persistence**: Variables and files

## Step 1: Install Required Libraries

In [ ]:
!pip install -q -U transformers datasets peft trl bitsandbytes accelerate huggingface_hub
!nvidia-smi

## Step 2: Authenticate & Download Dataset from Hugging Face Hub

In [ ]:
import os
import json
from pathlib import Path
from huggingface_hub import login, hf_hub_download
from datasets import Dataset

HF_TOKEN = ""
login(token=HF_TOKEN, add_to_git_credential=True)

DATASET_REPO = "krishna1707/buildfail-dataset"
print(f"Downloading dataset from: https://huggingface.co/datasets/{DATASET_REPO}...")

train_path = hf_hub_download(repo_id=DATASET_REPO, filename="train.jsonl", repo_type="dataset", token=HF_TOKEN)
val_path = hf_hub_download(repo_id=DATASET_REPO, filename="validation.jsonl", repo_type="dataset", token=HF_TOKEN)

PROMPT_TEMPLATE = """<|im_start|>system
You are BuildFail-SLM, an expert AI DevOps and software repair model.
Analyze CI/CD failures accurately, identify root causes, and propose non-destructive unified diffs.
<|im_end|>
<|im_start|>user
{instruction}

{input_text}
<|im_end|>
<|im_start|>assistant
{output_text}
<|im_end|>"""

def prepare_dataset(jsonl_path: str) -> Dataset:
    records = []
    with open(jsonl_path, "r", encoding="utf-8") as f:
        for line in f:
            d = json.loads(line)
            inst = d.get("instruction", "")
            inp = d.get("input", "")
            out = d.get("output", "")
            inp_str = inp if isinstance(inp, str) else json.dumps(inp, indent=2)
            out_str = out if isinstance(out, str) else json.dumps(out, indent=2)
            # Pre-truncate logs > 2500 chars to prevent 225k token overflow warning
            if len(inp_str) > 2500:
                inp_str = inp_str[:2500] + "\n...[truncated]"
            records.append({"text": PROMPT_TEMPLATE.format(
                instruction=inst, input_text=inp_str, output_text=out_str
            )})
    return Dataset.from_list(records)

train_ds = prepare_dataset(train_path)
val_ds = prepare_dataset(val_path)
print(f"✅ Successfully loaded {len(train_ds):,} training examples & {len(val_ds):,} validation examples!")


## Step 3: Load Qwen2.5-Coder-3B in 4-bit NF4 with Dual GPU Auto-Mapping

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

BASE_MODEL_NAME = "Qwen/Qwen2.5-Coder-3B-Instruct"

# 4-bit NormalFloat with FP16 compute for Tesla T4 Tensor Cores
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.float16
)

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL_NAME, token=HF_TOKEN, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.model_max_length = 512

print(f"Loading {BASE_MODEL_NAME} in 4-bit across available GPUs...")
model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL_NAME,
    token=HF_TOKEN,
    quantization_config=bnb_config,
    torch_dtype=torch.float16,
    device_map="auto",
    trust_remote_code=True
)

model = prepare_model_for_kbit_training(model, use_gradient_checkpointing=True)

# Ensure all adapter parameters are Float16
for param in model.parameters():
    if param.dtype == torch.bfloat16:
        param.data = param.data.to(torch.float16)

peft_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"]
)

model = get_peft_model(model, peft_config)
model.print_trainable_parameters()
print("Base model + LoRA adapter successfully allocated across Kaggle GPUs!")


## Step 4: High-Throughput SFT Fine-Tuning Loop

In [ ]:
from trl import SFTTrainer, SFTConfig
import trl.trainer.sft_trainer

# 1. Disable TRL's experimental chunked CE patch for multi-GPU partial objects
trl.trainer.sft_trainer._patch_chunked_ce_lm_head = lambda *args, **kwargs: None

# 2. Robust Trainer subclass: uses standard PyTorch CausalLM cross-entropy loss
# (bypasses TRL's broken internal entropy / num_valid_tokens tracking)
class RobustSFTTrainer(SFTTrainer):
    def compute_loss(self, model, inputs, return_outputs=False, num_items_in_batch=None):
        outputs = model(**inputs)
        loss = outputs.loss
        if loss is not None and loss.dim() > 0:
            loss = loss.mean()
        return (loss, outputs) if return_outputs else loss

# Optimized for Kaggle 32 GB VRAM (Batch size 4 per device allows 2x faster throughput)
training_args = SFTConfig(
    output_dir="./checkpoints",
    per_device_train_batch_size=4,       # Takes advantage of 32 GB VRAM
    gradient_accumulation_steps=4,      # Effective batch size = 16
    learning_rate=2e-4,
    num_train_epochs=1,                 # Full epoch across all 32,879 samples
    logging_steps=25,
    save_strategy="steps",
    save_steps=500,
    eval_strategy="steps",
    eval_steps=500,
    fp16=False,                         # Pure FP16 compute via bitsandbytes without GradScaler crash
    bf16=False,
    optim="paged_adamw_8bit",
    gradient_checkpointing=True,
    max_grad_norm=0.3,
    warmup_steps=100,
    lr_scheduler_type="cosine",
    dataset_text_field="text",
    max_length=512,
    report_to="none"
)

trainer = RobustSFTTrainer(
    model=model,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    processing_class=tokenizer,
    args=training_args
)

print("=" * 70)
print(f"COMMENCING HIGH-THROUGHPUT TRAINING ON KAGGLE ({len(train_ds):,} SAMPLES)")
print("=" * 70)
trainer.train()


## Step 5: Save & Push Fine-Tuned Adapter to Hugging Face Hub

In [ ]:
OUTPUT_MODEL_REPO = "krishna1707/buildfail-slm-adapter"

print(f"Pushing trained adapter to Hugging Face: https://huggingface.co/{OUTPUT_MODEL_REPO}...")
trainer.model.push_to_hub(OUTPUT_MODEL_REPO, token=HF_TOKEN, private=False)
tokenizer.push_to_hub(OUTPUT_MODEL_REPO, token=HF_TOKEN, private=False)
print("🎉 Successfully published BuildFail-SLM Adapter to Hugging Face Model Hub!")


## Step 6: Test & Verify Model Output

In [ ]:
test_prompt = PROMPT_TEMPLATE.format(
    instruction="Diagnose the failure root cause and propose a patch for this Docker CI build:",
    input_text="Step 4/10 : RUN pip install -r requirements.txt\nERROR: Could not find a version that satisfies the requirement torch==2.6.0+cu124\nERROR: No matching distribution found for torch==2.6.0+cu124",
    output_text=""
).strip()

inputs = tokenizer(test_prompt, return_tensors="pt").to("cuda")
with torch.no_grad():
    outputs = model.generate(**inputs, max_new_tokens=150, temperature=0.2, top_p=0.9)
output_text = tokenizer.decode(outputs[0][inputs.input_ids.shape[1]:], skip_special_tokens=True)
print("\n--- Model Output ---\n")
print(output_text)
